# IV_02 — Excel y Power BI en la cadena de valor

## 1. Objetivo

Diferenciar el rol de Excel y Power BI, y generar un dataset listo para importar en Power BI.

## 2. Concepto

**Excel** es ideal para modelos de ingeniería puntuales (curvas de bomba, balances). **Power BI** consolida datos de múltiples fuentes para dashboards de jefatura.

### Si vienes de Excel...
Una hoja con columnas Fecha | Tag | Valor es equivalente al CSV que importarás en Power BI.

In [ ]:
import os
import sqlite3
from pathlib import Path

import pandas as pd

MOD_DIR = Path.cwd()
os.chdir(MOD_DIR)
DATA_DIR = MOD_DIR / "data"
OUTPUT_DIR = MOD_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
DB_PATH = DATA_DIR / "ecosistema_local.db"


## 3. Generar dataset para Power BI

In [ ]:
# Cargar export PI simulado
df_pi = pd.read_csv(DATA_DIR / "lecturas_pi_historiador.csv", parse_dates=["Timestamp"])
print(f"Registros PI: {len(df_pi)}")

# Agregar diario para dashboard (menos filas, más legible)
df_daily = (
    df_pi[df_pi["Quality"] == "GOOD"]
    .pivot_table(index="Timestamp", columns="Tag", values="Value", aggfunc="mean")
    .resample("1D")
    .mean()
    .reset_index()
)
df_daily["Timestamp"] = df_daily["Timestamp"].dt.strftime("%Y-%m-%d")

# Formato largo para Power BI
dashboard = df_daily.melt(id_vars=["Timestamp"], var_name="Tag", value_name="Valor")
dashboard["Equipo"] = dashboard["Tag"].str.split(".").str[0]

pbi_path = DATA_DIR / "powerbi" / "dashboard_fuente.csv"
pbi_path.parent.mkdir(exist_ok=True)
dashboard.to_csv(pbi_path, index=False)
print(f"Dataset Power BI: {pbi_path}")
dashboard.head()


## 4. KPIs que podrías calcular en Excel o Power BI

| KPI | Fórmula conceptual |
|-----|-------------------|
| Disponibilidad | MTBF / (MTBF + MTTR) |
| Eficiencia bomba | Caudal medido / Caudal diseño |
| Vibración promedio | PROMEDIO(lecturas 24h) |

In [ ]:
# KPI ejemplo: vibración promedio PUMP101
vib = df_pi[(df_pi["Tag"] == "PUMP101.VIBRATION_RMS") & (df_pi["Quality"] == "GOOD")]
kpi_vib = vib["Value"].mean()
print(f"KPI Vibración PUMP101: {kpi_vib:.2f} mm/s")


## 5. Práctica Power BI (manual, 10 min)

Sigue las instrucciones en `powerbi/instrucciones_dashboard.md`:
1. Importar `data/powerbi/dashboard_fuente.csv`
2. Crear gráfico de tendencia
3. Crear tarjeta KPI de vibración

## 6. Resumen y siguiente paso

- Excel: modelado local; Power BI: visualización corporativa.
- Python prepara y exporta datos limpios para ambos.
- El CSV generado es la interfaz entre Python y Power BI.

**Siguiente:** `IV_03_sql_supabase.ipynb`